In [1]:
import torch
import torch.nn as nn
import math

torch.manual_seed(1)


In [2]:
image = torch.randn(2, 3, 224, 224)

In [3]:
patch_size = 16
embedding_size = 64

patch_layer = nn.Conv2d(
    3, embedding_size,
    kernel_size=patch_size,
    stride=patch_size
)

patches = patch_layer(image)


In [5]:
patches = patches.flatten(2)
patches = patches.transpose(1, 2)

print("Patch sequence shape:", patches.shape)

Patch sequence shape: torch.Size([2, 64, 196])


In [7]:
class_token = torch.randn(1, 1, embedding_size)
class_token = class_token.expand(2, -1, -1)


patches = patches.transpose(1, 2)

tokens = torch.cat([class_token, patches], dim=1)

In [8]:
position = torch.randn(1, tokens.shape[1], embedding_size)

tokens = tokens + position


In [9]:
heads = 4
head_size = embedding_size // heads

q_layer = nn.Linear(embedding_size, embedding_size)
k_layer = nn.Linear(embedding_size, embedding_size)
v_layer = nn.Linear(embedding_size, embedding_size)

Q = q_layer(tokens)
K = k_layer(tokens)
V = v_layer(tokens)

Q = Q.view(2, -1, heads, head_size).transpose(1, 2)
K = K.view(2, -1, heads, head_size).transpose(1, 2)
V = V.view(2, -1, heads, head_size).transpose(1, 2)

attention = Q @ K.transpose(-2, -1)
attention = attention / math.sqrt(head_size)

attention = torch.softmax(attention, dim=-1)

output = attention @ V

output = output.transpose(1, 2)
output = output.reshape(2, -1, embedding_size)

print("Attention output shape:", output.shape)


Attention output shape: torch.Size([2, 197, 64])
